# Gaussian Copula

First of the four generation notebooks. Each follows the same shape: load the shared
training data, search for a configuration, then generate and save the synthetic data. One
method per notebook means each gets identical treatment, and a problem with one cannot
interrupt the others.

This notebook produces two datasets rather than one. The first uses the library defaults
and is saved as the default dataset. The second uses the configuration chosen by the
search below and is saved as the tuned dataset. Both are the size of the training set, and
each is recorded in the shared timing log under its own name, with its settings and the
hardware it ran on.

Those two datasets are the project's two experiments. The first asks how the four methods
compare out of the box, which is what a practitioner would get by installing a library and
following its documentation. The second asks whether that comparison survives tuning, so
that a method is not judged solely on how well its authors chose its defaults. Running
this notebook from top to bottom produces both.

Gaussian Copula is the statistical baseline. It learns the shape of each column and the
correlations between them, then samples from those shapes in a way that respects the
correlations. No neural networks are involved, which makes it much the fastest of the four
and a useful reference point: if the more complex methods cannot beat it, their extra cost
is hard to justify.

Its known limitation, covered in the literature review, is that correlation only captures
straight-line relationships. More complicated interactions can be missed, and the
evaluation will show whether that matters for this cohort.

## Setup

Gaussian Copula comes from sdv, which also provides CTGAN and TVAE. No GPU is needed; this
method fits in seconds.

In [1]:
%pip install -q pandas pyarrow sdv

## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and delete
the data once the work is finished.

In [2]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/mimic-synthetic-pipeline


## Training data

Only the training set is loaded. The generator must never see the test set, which is what
the evaluation later uses to judge whether synthetic-trained models generalise to unseen
patients.

In [3]:
from pathlib import Path

import pandas as pd

if not Path("data/train.parquet").exists():
    raise FileNotFoundError(
        "data/train.parquet not found. Run the extraction and preparation steps first."
    )

TARGET = "readmitted_30d"
train_df = pd.read_parquet("data/train.parquet")
print(f"Training data: {len(train_df):,} admissions, readmission rate {train_df[TARGET].mean():.4f}")

Training data: 427,354 admissions, readmission rate 0.2056


## Hyperparameter search

Library defaults are one particular choice, made by the people who wrote the library
rather than for this data. This section asks whether Gaussian Copula does better under
other settings, so that the comparison between methods is a comparison of the methods and
not of how well each was configured out of the box.

### Method

The search uses successive halving. Every configuration is screened on a small sample at a
reduced budget, and only the strongest few are promoted to a larger sample, the full
budget and repeated seeds. Configurations that look unpromising are therefore abandoned
early, which is where the saving in computation comes from, and the survivors are assessed
with enough repetition that the choice between them is not made on a single noisy run.

Neither library used in this project supports resuming training or validation-based early
stopping at a practical cost, so stopping is applied at the level of the configuration
rather than the epoch. The convergence rule used elsewhere in this project is applied to
each trial's loss curve and reported alongside its score, so a configuration that had not
finished training is visible rather than silently accepted.

Selection uses utility measured on the validation split. The test set is never involved,
so no part of the search can influence the reported results. Fidelity is recorded for
every trial but is not optimised, which means any movement in it is a consequence of
selecting for utility rather than a target of the search. That relationship is itself a
finding worth reporting.

Configurations whose mean scores fall within 0.005 of the best are reported as
indistinguishable, following the same reasoning applied to the differences between
methods. Where several are tied, the simplest should be preferred.

### Parameters varied

Both parameters the synthesizer exposes. `default_distribution` forces a single marginal
family across every column, and `numerical_distributions` assigns them per column. The
distinction matters here because length of stay is heavily right-skewed while age is
bounded and capped at 91, so a single family is unlikely to suit both.

### Cost and outputs

Nine configurations. The copula fits in seconds, so the whole search completes in a few
minutes.

Three files are written: every individual trial, a summary averaged over seeds, and the
selected configuration as JSON. The training cell below reads the selected configuration
automatically and applies it to the full training split. Nothing here overwrites the saved
synthetic datasets.

In [4]:
import json
import time

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Search budget. Reduce these if the search needs to finish sooner. ----------------
SCREEN_ROWS = 25_000     # rows per trial in the screening round
SCREEN_FRACTION = 0.33   # fraction of the full training budget used when screening
PROMOTE_ROWS = 100_000   # rows per trial once a configuration is promoted
PROMOTE_KEEP = 3         # configurations carried into the promotion round
PROMOTE_SEEDS = (0, 1, 2)  # repeats per promoted configuration
TIE_THRESHOLD = 0.005    # ROC AUC difference treated as indistinguishable
RESUME_SCREENING = True  # reuse a completed screening round rather than repeating it
FIDELITY_TOLERANCE = 1.5 # a candidate may not worsen KS or TVD beyond this multiple of
                         # the library default, however much utility it gains

NUMERIC_COLS = ["age_at_admission", "length_of_stay_days"]
CATEGORICAL_COLS = [
    "gender", "admission_type", "admission_location", "insurance",
    "marital_status", "race", "language", "had_icu_stay",
]

for _required in ["data/train_fit.parquet", "data/train_val.parquet"]:
    if not Path(_required).exists():
        raise FileNotFoundError(
            f"{_required} not found. Re-run 02_data_preparation.ipynb, which writes the "
            "validation split this search depends on."
        )

fit_df = pd.read_parquet("data/train_fit.parquet")
val_df = pd.read_parquet("data/train_val.parquet")
for _c in fit_df.columns:
    if str(fit_df[_c].dtype) == "Int64":
        fit_df[_c] = fit_df[_c].astype("int64")
        val_df[_c] = val_df[_c].astype("int64")

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

print(f"Fitting split {len(fit_df):,} rows | validation split {len(val_df):,} rows")


def make_classifier() -> Pipeline:
    """The classifier from the main evaluation, so scores are directly comparable."""
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])


def utility_on_validation(synthetic: pd.DataFrame) -> float:
    """Train on synthetic, score on the real validation split. The test set is never used."""
    clf = make_classifier()
    clf.fit(synthetic.drop(columns=[TARGET]), synthetic[TARGET])
    proba = clf.predict_proba(val_df.drop(columns=[TARGET]))[:, 1]
    return float(roc_auc_score(val_df[TARGET], proba))


def _tvd(real_col: pd.Series, syn_col: pd.Series) -> float:
    p = real_col.value_counts(normalize=True)
    q = syn_col.value_counts(normalize=True)
    return 0.5 * sum(abs(p.get(c, 0.0) - q.get(c, 0.0)) for c in p.index.union(q.index))


def fidelity_summary(synthetic: pd.DataFrame, real: pd.DataFrame) -> tuple:
    ks = float(np.mean([
        ks_2samp(real[c].astype(float), synthetic[c].astype(float)).statistic
        for c in NUMERIC_COLS
    ]))
    tvd = float(np.mean([
        _tvd(real[c].astype(str), synthetic[c].astype(str)) for c in CATEGORICAL_COLS
    ]))
    return ks, tvd


def converged(loss_series) -> tuple:
    """The stopping rule used elsewhere: mean loss over the final fifth of training
    against the fifth before it. Returns the percentage improvement and a flag."""
    if loss_series is None or len(loss_series) < 10:
        return float("nan"), None
    values = np.asarray(loss_series, dtype=float)
    n = len(values)
    previous = values[int(n * 0.6):int(n * 0.8)].mean()
    final = values[int(n * 0.8):].mean()
    improvement = (previous - final) / abs(previous) * 100
    return float(improvement), bool(improvement <= 1.0)


def set_seed(seed: int) -> None:
    """The sdv synthesizers expose no seed argument, so the global generators are set."""
    np.random.seed(seed)
    torch.manual_seed(seed)


def _screen(space, screen_sample, screen_budget, fit_and_sample):
    """Round one. Every configuration, one seed, reduced sample and budget."""
    screened = []
    for i, (label, config) in enumerate(space, start=1):
        print(f"\n[{i}/{len(space)}] {label}", flush=True)
        t0 = time.time()
        try:
            set_seed(0)
            synthetic, losses = fit_and_sample(config, screen_sample, screen_budget)
            auc = utility_on_validation(synthetic)
            ks, tvd = fidelity_summary(synthetic, screen_sample)
            improvement, is_converged = converged(losses)
            seconds = time.time() - t0
            screened.append({
                "config_label": label, "config": json.dumps(config), "round": "screen",
                "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                "loss_improvement_pct": improvement, "converged": is_converged,
                "seconds": seconds,
            })
            flag = "" if is_converged is None else ("converged" if is_converged else "NOT converged")
            print(f"      ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f} | "
                  f"{seconds:.0f}s {flag}", flush=True)
        except Exception as exc:  # noqa: BLE001
            print(f"      failed: {type(exc).__name__}: {exc}", flush=True)
    return screened


def run_search(space, fit_and_sample, method_slug, full_budget):
    """Successive halving. Every configuration is screened on a small sample at a reduced
    budget; the best few are promoted to a larger sample, a full budget and repeated seeds.
    Unpromising configurations are therefore stopped early, which is where the compute
    saving comes from."""
    screen_sample = fit_df.sample(n=min(SCREEN_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    screen_budget = max(1, int(full_budget * SCREEN_FRACTION))
    screen_path = out_dir / f"tuning_{method_slug}_screen.csv"

    # Screening is written to disk as soon as it completes and reused if this cell is run
    # again. A session that drops during promotion therefore costs only the promotion
    # round, which matters because screening the diffusion model takes several hours.
    if RESUME_SCREENING and screen_path.exists():
        screen_df = pd.read_csv(screen_path)
        print(f"Reusing the screening round from {screen_path}")
        print(f"  {len(screen_df)} configurations already scored. Delete that file to "
              f"screen again.", flush=True)
        screened = screen_df.to_dict("records")
    else:
        print(f"\n{'=' * 78}")
        print(f"ROUND 1, screening {len(space)} configurations")
        print(f"{len(screen_sample):,} rows, budget {screen_budget}, one seed each")
        print(f"{'=' * 78}", flush=True)
        screened = _screen(space, screen_sample, screen_budget, fit_and_sample)
        if screened:
            pd.DataFrame(screened).to_csv(screen_path, index=False)
            print(f"\nScreening saved to {screen_path}", flush=True)

    if not screened:
        raise RuntimeError("Every configuration failed during screening.")

    screen_df = pd.DataFrame(screened).sort_values("val_roc_auc", ascending=False)

    # Utility alone is not a sufficient selection criterion. A configuration can raise the
    # downstream score while badly degrading the distributions, which would be a poor
    # outcome for a project whose argument is that these dimensions must be read together.
    # Candidates are therefore restricted to those whose fidelity is no worse than the
    # first configuration in the search space, the library default, by more than the
    # tolerance below. Utility decides the ranking within that set.
    baseline_label = space[0][0]
    baseline = screen_df[screen_df["config_label"] == baseline_label]
    eligible = screen_df
    if not baseline.empty:
        base_ks = float(baseline.iloc[0]["mean_ks"])
        base_tvd = float(baseline.iloc[0]["mean_tvd"])
        limit_ks = base_ks * FIDELITY_TOLERANCE + 1e-6
        limit_tvd = base_tvd * FIDELITY_TOLERANCE + 1e-6
        eligible = screen_df[(screen_df["mean_ks"] <= limit_ks)
                             & (screen_df["mean_tvd"] <= limit_tvd)]
        excluded = screen_df[~screen_df["config_label"].isin(eligible["config_label"])]
        print()
        print(f"Fidelity guardrail: KS <= {limit_ks:.4f}, TVD <= {limit_tvd:.4f}")
        print(f"  ({FIDELITY_TOLERANCE}x the baseline configuration '{baseline_label}')")
        if len(excluded):
            print(f"  Excluded {len(excluded)} configuration(s) that raised utility at the "
                  f"cost of fidelity:")
            for _, r in excluded.iterrows():
                print(f"    {r['config_label']:<42} ROC AUC {r['val_roc_auc']:.4f}  "
                      f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}")
        if eligible.empty:
            print("  No configuration met the guardrail, so it has been relaxed for this run.")
            eligible = screen_df

    promoted = eligible.head(PROMOTE_KEEP)

    print(f"\n{'=' * 78}")
    print(f"ROUND 2, promoting the top {len(promoted)} of {len(screen_df)}")
    print(f"{min(PROMOTE_ROWS, len(fit_df)):,} rows, budget {full_budget}, "
          f"{len(PROMOTE_SEEDS)} seeds each")
    print(f"{'=' * 78}", flush=True)

    promote_sample = fit_df.sample(n=min(PROMOTE_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    promoted_rows = []
    for i, row in enumerate(promoted.itertuples(index=False), start=1):
        config = json.loads(row.config)
        print(f"\n[{i}/{len(promoted)}] {row.config_label}", flush=True)
        for seed in PROMOTE_SEEDS:
            t0 = time.time()
            try:
                set_seed(seed)
                synthetic, losses = fit_and_sample(config, promote_sample, full_budget, seed=seed)
                auc = utility_on_validation(synthetic)
                ks, tvd = fidelity_summary(synthetic, promote_sample)
                improvement, is_converged = converged(losses)
                promoted_rows.append({
                    "config_label": row.config_label, "config": row.config, "round": "promote",
                    "seed": seed, "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                    "loss_improvement_pct": improvement, "converged": is_converged,
                    "seconds": time.time() - t0,
                })
                print(f"      seed {seed}: ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f}", flush=True)
            except Exception as exc:  # noqa: BLE001
                print(f"      seed {seed} failed: {type(exc).__name__}: {exc}", flush=True)

    promote_df = pd.DataFrame(promoted_rows)
    summary = (
        promote_df.groupby(["config_label", "config"])
        .agg(mean_roc_auc=("val_roc_auc", "mean"), sd_roc_auc=("val_roc_auc", "std"),
             mean_ks=("mean_ks", "mean"), mean_tvd=("mean_tvd", "mean"),
             runs=("val_roc_auc", "size"))
        .reset_index().sort_values("mean_roc_auc", ascending=False)
    )

    pd.concat([screen_df, promote_df], ignore_index=True).to_csv(
        out_dir / f"tuning_{method_slug}_trials.csv", index=False)
    summary.to_csv(out_dir / f"tuning_{method_slug}_summary.csv", index=False)

    print(f"\n{'=' * 78}")
    print("PROMOTION RESULTS, mean over seeds")
    print(f"{'=' * 78}")
    for _, r in summary.iterrows():
        sd = 0.0 if pd.isna(r["sd_roc_auc"]) else r["sd_roc_auc"]
        print(f"  {r['config_label']:<46} {r['mean_roc_auc']:.4f} +/- {sd:.4f}  "
              f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}  (n={r['runs']})")

    # Utility decides the ranking, but configurations within TIE_THRESHOLD of the leader
    # are not separable on this evidence. Rather than taking whichever happened to score
    # highest, the tie is broken on fidelity: lower distributional error first, and the
    # library default preferred over an equally faithful alternative.
    leader = summary.iloc[0]["mean_roc_auc"]
    tied = summary[summary["mean_roc_auc"] >= leader - TIE_THRESHOLD].copy()
    tied["is_default"] = tied["config"] == "{}"
    tied = tied.sort_values(
        ["mean_ks", "mean_tvd", "is_default"], ascending=[True, True, False]
    )
    best = tied.iloc[0]

    checkpoint = {
        "method": method_slug,
        "selected_config": json.loads(best["config"]),
        "selected_label": best["config_label"],
        "mean_val_roc_auc": float(best["mean_roc_auc"]),
        "sd_val_roc_auc": None if pd.isna(best["sd_roc_auc"]) else float(best["sd_roc_auc"]),
        "seeds": list(PROMOTE_SEEDS),
        "screen_rows": int(min(SCREEN_ROWS, len(fit_df))),
        "promote_rows": int(min(PROMOTE_ROWS, len(fit_df))),
        "full_budget": full_budget,
        "tied_within_threshold": tied["config_label"].tolist(),
    }
    with open(out_dir / f"tuning_{method_slug}_selected.json", "w", encoding="utf-8") as fh:
        json.dump(checkpoint, fh, indent=2)

    print()
    print(f"Selected: {best['config_label']}")
    if len(tied) > 1:
        print(f"{len(tied)} configurations fell within {TIE_THRESHOLD} ROC AUC of the leader "
              f"and are not separable on utility:")
        for _, r in tied.iterrows():
            print(f"    {r['config_label']:<44} ROC AUC {r['mean_roc_auc']:.4f}  "
                  f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}")
        print("The tie was broken on fidelity, preferring the lowest distributional error.")
        print("Report the tie rather than presenting the winner as clearly better.")
    print(f"\nSaved outputs/tuning_{method_slug}_selected.json for the final run.")
    return summary


METHOD_SLUG = "gaussian_copula"
FULL_BUDGET = 1  # the copula has no iterative budget; kept for interface consistency

from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

# Both levers the synthesizer exposes. default_distribution forces one marginal family
# across every column; numerical_distributions sets them per column, which matters here
# because length of stay is heavily right-skewed while age is bounded and capped at 91.
SEARCH_SPACE = [
    ("library defaults", {}),
    ("default=norm", {"default_distribution": "norm"}),
    ("default=beta", {"default_distribution": "beta"}),
    ("default=truncnorm", {"default_distribution": "truncnorm"}),
    ("default=gamma", {"default_distribution": "gamma"}),
    ("default=uniform", {"default_distribution": "uniform"}),
    ("per-column: gamma LoS, truncnorm age", {"numerical_distributions": {
        "length_of_stay_days": "gamma", "age_at_admission": "truncnorm"}}),
    ("per-column: beta LoS, norm age", {"numerical_distributions": {
        "length_of_stay_days": "beta", "age_at_admission": "norm"}}),
    ("per-column: gamma LoS, beta age", {"numerical_distributions": {
        "length_of_stay_days": "gamma", "age_at_admission": "beta"}}),
]


def fit_and_sample(config, data, budget, seed=0):
    metadata = Metadata.detect_from_dataframe(data, table_name="cohort")
    model = GaussianCopulaSynthesizer(metadata, **config)
    model.fit(data)
    return model.sample(num_rows=len(data)), None

summary = run_search(SEARCH_SPACE, fit_and_sample, METHOD_SLUG, FULL_BUDGET)
summary


Fitting split 342,004 rows | validation split 85,350 rows

ROUND 1, screening 9 configurations
25,000 rows, budget 1, one seed each

[1/9] library defaults


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


      ROC AUC 0.5511 | KS 0.0873 | TVD 0.0047 | 7s 

[2/9] default=norm


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


      ROC AUC 0.5139 | KS 0.1503 | TVD 0.0804 | 3s 

[3/9] default=beta


...

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


      seed 1: ROC AUC 0.5434 | KS 0.0518 | TVD 0.0021


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


      seed 2: ROC AUC 0.5434 | KS 0.0518 | TVD 0.0021

PROMOTION RESULTS, mean over seeds
  per-column: beta LoS, norm age                 0.5434 +/- 0.0000  KS 0.0518  TVD 0.0021  (n=3)
  default=beta                                   0.5429 +/- 0.0000  KS 0.0764  TVD 0.0027  (n=3)
  library defaults                               0.5429 +/- 0.0000  KS 0.0764  TVD 0.0027  (n=3)

Selected: per-column: beta LoS, norm age
Within 0.005 of the best, so not separable on this evidence:
    per-column: beta LoS, norm age
    default=beta
    library defaults
Prefer the simplest of these, and say so in the write-up.

Saved outputs/tuning_gaussian_copula_selected.json for the final run.


,config_label,config,mean_roc_auc,sd_roc_auc,mean_ks,mean_tvd,runs
2,"per-column: beta LoS, norm age","{""numerical_distributions"": {""length_of_stay_d...",0.543446,0.0,0.051830,0.002095,3
0,default=beta,"{""default_distribution"": ""beta""}",0.542897,0.0,0.076365,0.002710,3
1,library defaults,{},0.542897,0.0,0.076365,0.002710,3


## Generating both datasets

Two datasets are produced from this method, not one. The first uses the library defaults,
the second the configuration chosen by the search above. Both are generated in this
notebook so that a single run evidences both experiments, and so the comparison between
them is made on identical data, an identical split and identical evaluation code.

Each is written under its own filename and recorded as its own row in the shared timing
log, tagged with the configuration and the hardware. The evaluation notebook picks up
whichever datasets exist, so it reports both without further intervention.

The two runs are in separate cells deliberately. Training is the expensive part, and a
failure in the second should not discard the first.

The timing log records whichever accelerator the session had attached, which was the same
one used for the three neural methods. Gaussian Copula does not use it. This method is
fitted on the CPU, so its timings are CPU timings and should be described that way rather
than compared directly with GPU training times.

In [5]:
METHOD_NAME = "Gaussian Copula"
METHOD_SLUG = "gaussian_copula"

import json as _json
import time

import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

try:
    import torch as _torch
    GPU_NAME = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else "CPU"
except Exception:  # noqa: BLE001
    GPU_NAME = "unknown"

# The tuned configuration comes from the search above. If it has not been run, only the
# default experiment is possible and the tuned cell will say so rather than failing.
TUNED_CONFIG = None
TUNED_LABEL = None
_selected = Path(f"outputs/tuning_{METHOD_SLUG}_selected.json")
if _selected.exists():
    with open(_selected, encoding="utf-8") as _fh:
        _payload = _json.load(_fh)
    TUNED_CONFIG = _payload["selected_config"]
    TUNED_LABEL = _payload["selected_label"]
    print(f"Tuned configuration available: {TUNED_LABEL}")
    print(f"  {TUNED_CONFIG}")
else:
    print(f"No search result at {_selected}. Only the default experiment can be run.")

# Each completed run is written to its own file the moment it finishes. Reloading them
# here means a notebook restarted after a dropped session picks up where it stopped
# instead of repeating an experiment that already produced its dataset.
RUN_DIR = out_dir / "runs"
RUN_DIR.mkdir(exist_ok=True)

results = {}
for _p in sorted(RUN_DIR.glob(f"{METHOD_SLUG}_*.json")):
    with open(_p, encoding="utf-8") as _fh:
        _rec = _json.load(_fh)
    if Path(f"data/synthetic_{METHOD_SLUG}_{_rec['experiment']}.parquet").exists():
        results[_rec["experiment"]] = _rec
if results:
    print(f"Recovered from an earlier session: {', '.join(sorted(results))}")


def convergence(losses, n_label):
    """The stopping rule used throughout: mean loss over the final fifth of training
    against the fifth before it."""
    if losses is None or len(losses) < 10:
        return None, None
    values = np.asarray(losses, dtype=float)
    n = len(values)
    previous = values[int(n * 0.6):int(n * 0.8)].mean()
    final = values[int(n * 0.8):].mean()
    improvement = (previous - final) / abs(previous) * 100
    print(f"  convergence: {previous:.4f} -> {final:.4f}, {improvement:+.2f}% across the "
          f"final fifth ({'flat' if improvement <= 1.0 else 'still improving'}, {n_label})")
    return float(improvement), bool(improvement <= 1.0)


def sanity(synthetic):
    checks = pd.DataFrame({
        "statistic": ["Readmission rate", "Mean age", "Mean length of stay (days)"],
        "real": [
            round(train_df[TARGET].mean(), 4),
            round(train_df["age_at_admission"].astype(float).mean(), 1),
            round(train_df["length_of_stay_days"].mean(), 2),
        ],
        "synthetic": [
            round(synthetic[TARGET].mean(), 4),
            round(synthetic["age_at_admission"].astype(float).mean(), 1),
            round(synthetic["length_of_stay_days"].mean(), 2),
        ],
    })
    print(checks.to_string(index=False))
    return checks


def run_experiment(label, config, slug):
    """Fit, generate, save and summarise one configuration."""
    path = f"data/synthetic_{METHOD_SLUG}_{slug}.parquet"
    record_path = RUN_DIR / f"{METHOD_SLUG}_{slug}.json"

    # Reuse a run that already produced both its record and its dataset, so re-running
    # this cell after an interruption costs nothing.
    if record_path.exists() and Path(path).exists():
        with open(record_path, encoding="utf-8") as fh:
            record = _json.load(fh)
        results[slug] = record
        print(f"{METHOD_NAME}: {label} already complete, reusing {path}")
        print(f"  training {record['train_seconds']:.1f}s, generation "
              f"{record['generate_seconds']:.1f}s, {record['rows_generated']:,} rows")
        return record

    print("=" * 78)
    print(f"{METHOD_NAME}: {label}")
    print("=" * 78, flush=True)

    synthetic, train_seconds, generate_seconds, losses = fit_and_generate(config, slug)

    synthetic.to_parquet(path, index=False)

    improvement, converged = convergence(losses, label)
    if losses is not None:
        try:
            frame = pd.DataFrame({"loss": np.asarray(losses, dtype=float)})
            frame.to_csv(out_dir / f"loss_history_{METHOD_SLUG}_{slug}.csv", index=False)
        except Exception as exc:  # noqa: BLE001
            print(f"  could not write the loss history: {type(exc).__name__}")

    print(f"  training {train_seconds:.1f}s, generation {generate_seconds:.1f}s, "
          f"{len(synthetic):,} rows -> {path}")
    sanity(synthetic)

    record = {
        "method": METHOD_NAME,
        "experiment": slug,
        "config": label,
        "rows_generated": len(synthetic),
        "train_seconds": round(train_seconds, 1),
        "generate_seconds": round(generate_seconds, 1),
        "gpu": GPU_NAME,
        "loss_improvement_pct": improvement,
        "converged": converged,
    }
    results[slug] = record
    with open(record_path, "w", encoding="utf-8") as fh:
        _json.dump(record, fh, indent=2)
    return record


def fit_and_generate(config, slug=''):
    from sdv.metadata import Metadata
    from sdv.single_table import GaussianCopulaSynthesizer

    metadata = Metadata.detect_from_dataframe(train_df, table_name="cohort")
    model = GaussianCopulaSynthesizer(metadata, **config)

    t0 = time.time()
    model.fit(train_df)
    train_seconds = time.time() - t0

    t0 = time.time()
    synthetic = model.sample(num_rows=len(train_df))
    generate_seconds = time.time() - t0

    # The copula is fitted in closed form, so it has no loss curve to assess.
    return synthetic, train_seconds, generate_seconds, None


Tuned configuration available: per-column: beta LoS, norm age
  {'numerical_distributions': {'length_of_stay_days': 'beta', 'age_at_admission': 'norm'}}


In [6]:
run_experiment("library defaults", {}, "default")

Gaussian Copula: library defaults


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


  training 88.9s, generation 6.7s, 427,354 rows -> data/synthetic_gaussian_copula_default.parquet
                 statistic    real  synthetic
          Readmission rate  0.2056     0.2066
                  Mean age 58.8000    58.1000
Mean length of stay (days)  4.6600     4.4000


{'method': 'Gaussian Copula',
 'experiment': 'default',
 'config': 'library defaults',
 'rows_generated': 427354,
 'train_seconds': 88.9,
 'generate_seconds': 6.7,
 'gpu': 'NVIDIA A100-SXM4-40GB',
 'loss_improvement_pct': None,
 'converged': None}

In [7]:
if TUNED_CONFIG is None:
    print("No tuned configuration is available. Run the search above first.")
else:
    run_experiment(TUNED_LABEL, TUNED_CONFIG, "tuned")

Gaussian Copula: per-column: beta LoS, norm age


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


  training 81.6s, generation 6.2s, 427,354 rows -> data/synthetic_gaussian_copula_tuned.parquet
                 statistic    real  synthetic
          Readmission rate  0.2056      0.206
                  Mean age 58.8000     58.500
Mean length of stay (days)  4.6600      4.390


In [8]:
# Append both runs to the shared log, keyed on method and experiment so neither
# replaces the other.
log_path = out_dir / "generation_log.csv"
entries = pd.DataFrame(list(results.values()))

if log_path.exists():
    log = pd.read_csv(log_path)
    if "experiment" in log.columns:
        # Replace this method's rows for the experiments just run. Separately, drop any row
        # carrying no experiment, whichever method wrote it: those predate the
        # two-experiment structure, came from a superseded split, and would otherwise
        # persist unnoticed into the cost table.
        stale = (log["method"] == METHOD_NAME) & log["experiment"].isin(entries["experiment"])
        stale |= log["experiment"].isna()
        log = log[~stale]
    else:
        log = log[log["method"] != METHOD_NAME]
    log = pd.concat([log, entries], ignore_index=True)
else:
    log = entries

try:
    log.to_csv(log_path, index=False)
except Exception as exc:  # noqa: BLE001
    import csv as _csv
    with open(log_path, "w", newline="", encoding="utf-8") as fh:
        writer = _csv.writer(fh)
        writer.writerow([str(c) for c in log.columns])
        writer.writerows(log.itertuples(index=False, name=None))
    print(f"wrote the log via the standard library ({type(exc).__name__})")

print()
print("=" * 78)
print(f"{METHOD_NAME}: both experiments complete")
print("=" * 78)
print(entries[["experiment", "config", "train_seconds", "generate_seconds", "gpu"]].to_string(index=False))
if len(results) == 2:
    d, t = results["default"], results["tuned"]
    delta = t["train_seconds"] + t["generate_seconds"] - d["train_seconds"] - d["generate_seconds"]
    print(f"\nThe tuned configuration cost {delta:+.1f}s in total compute.")
    print("Whether it bought anything is decided in 07_evaluation.ipynb, not here.")
log



Gaussian Copula: both experiments complete
experiment                         config  train_seconds  generate_seconds                   gpu
   default               library defaults           88.9               6.7 NVIDIA A100-SXM4-40GB
     tuned per-column: beta LoS, norm age           81.6               6.2 NVIDIA A100-SXM4-40GB

The tuned configuration cost -7.8s in total compute.
Whether it bought anything is decided in 07_evaluation.ipynb, not here.


,method,rows_generated,train_seconds,generate_seconds,gpu,config,experiment,loss_improvement_pct,converged
0,CTGAN,427408,6724.9,5.5,NaN,NaN,NaN,NaN,NaN
1,TVAE,427408,4223.3,3.1,NaN,NaN,NaN,NaN,NaN
2,TabDDPM,427408,9809.1,264.0,NaN,NaN,NaN,NaN,NaN
3,Gaussian Copula,427354,88.9,6.7,NVIDIA A100-SXM4-40GB,library defaults,default,None,None
4,Gaussian Copula,427354,81.6,6.2,NVIDIA A100-SXM4-40GB,"per-column: beta LoS, norm age",tuned,None,None
